<a href="https://colab.research.google.com/github/harneet862/ur2phd-assignment/blob/main/part2_deberta_seed13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: System A, fine-tuned DeBERTa-v3-base (seed 13)
This notebook is identical for every seed. Only `SEED` changes. Each seed runs in its own notebook so a fresh session always starts from a fresh model.

## Setup

In [1]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn
!git clone https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git

Cloning into 'SemEval-2027-Task-2-DiCo-NLI'...
remote: Enumerating objects: 518, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 518 (delta 45), reused 42 (delta 42), pack-reused 471 (from 1)
Receiving objects: 100% (518/518), 29.85 MiB | 10.70 MiB/s, done.
Resolving deltas: 100% (268/268), done.


In [2]:
import pandas as pd, os

TASK_DIR = "/content/SemEval-2027-Task-2-DiCo-NLI"
TRAIN_PATH = f"{TASK_DIR}/final_data/train/dico_nli_train_track1_participant_labeled.csv"
DEV_REF_PATH = f"{TASK_DIR}/final_data/dev/dico_nli_dev_track1_reference.csv"
RESULTS_DIR = f"{TASK_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
SEED = 13

## Labels

In [4]:
LABELS = ["FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "EQUIVALENCE", "NEGATIVE_OTHER"]

label2id = dict()
id2label = dict()
for idx, elem in enumerate(LABELS):
    label2id[elem] = idx
    id2label[idx] = elem
print(label2id)

{'FORWARD_ENTAILMENT': 0, 'BACKWARD_ENTAILMENT': 1, 'EQUIVALENCE': 2, 'NEGATIVE_OTHER': 3}


## Data: train and dev

In [5]:
from datasets import Dataset

train = pd.read_csv(TRAIN_PATH)
train["labels"] = train["label"].map(label2id)
print("Train rows:", len(train), "| NaN labels:", train["labels"].isna().sum())

dev = pd.read_csv(DEV_REF_PATH)
dev["labels"] = dev["label"].map(label2id)
print("Dev rows:", len(dev), "| NaN labels:", dev["labels"].isna().sum())

Train rows: 3042 | NaN labels: 0
Dev rows: 660 | NaN labels: 0


In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

# both phrases go in as one sequence: [CLS] text1 [SEP] text2 [SEP]
def tokenize_pairs(batch):
    return tokeniser(batch["text1"], batch["text2"])

train_dataset = Dataset.from_pandas(train[["text1", "text2", "labels"]])
tokenized_train = train_dataset.map(tokenize_pairs, batched=True)

dev_dataset = Dataset.from_pandas(dev[["text1", "text2", "labels"]])
tokenized_dev = dev_dataset.map(tokenize_pairs, batched=True)

print(tokenized_train)
print(tokenized_dev)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/3042 [00:00<?, ? examples/s]

Map:   0%|          | 0/660 [00:00<?, ? examples/s]

Dataset({
    features: ['text1', 'text2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3042
})
Dataset({
    features: ['text1', 'text2', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 660
})


## Model (fresh for this seed)

In [7]:
import torch
from transformers import set_seed, AutoModelForSequenceClassification

set_seed(SEED)

# weights loaded in float32; fp16 mixed precision is handled by the Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.float32,
)
print(next(model.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier

torch.float32


## Trainer
Settings are fixed for all seeds: lr 2e-5, batch 16, 5 epochs, weight decay 0.01, no warmup, fp16, final checkpoint, no evaluation during training.

In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="deberta_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    seed=SEED,
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

collator = DataCollatorWithPadding(tokenizer=tokeniser)  # pads each batch to its longest item

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=collator,
    processing_class=tokeniser,
)

## Train
Expected: 955/955 steps, loss near 1.35 at step 50.

In [9]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,1.348454
100,1.169909
150,0.894433
200,0.679136
250,0.477411
300,0.439559
350,0.364138
400,0.326596
450,0.193450
500,0.246273


TrainOutput(global_step=955, training_loss=0.3766769526984679, metrics={'train_runtime': 167.0409, 'train_samples_per_second': 91.056, 'train_steps_per_second': 5.717, 'total_flos': 119118555450960.0, 'train_loss': 0.3766769526984679, 'epoch': 5.0})

In [10]:
# save the training loss log for this seed
loss_df = pd.DataFrame(trainer.state.log_history)
loss_df.to_csv(f"{RESULTS_DIR}/train_loss_seed{SEED}.csv", index=False)
print(loss_df[["step", "loss"]].dropna().to_string(index=False))

 step     loss
   50 1.348454
  100 1.169909
  150 0.894433
  200 0.679136
  250 0.477411
  300 0.439559
  350 0.364138
  400 0.326596
  450 0.193450
  500 0.246273
  550 0.202829
  600 0.163940
  650 0.122986
  700 0.125733
  750 0.113872
  800 0.092276
  850 0.066701
  900 0.082476
  950 0.080115


## Predict on dev and save

In [11]:
out = trainer.predict(tokenized_dev)
pred_ids = out.predictions.argmax(-1)
pred_labels = [id2label[int(i)] for i in pred_ids]

print(len(pred_labels))
print(pd.Series(pred_labels).value_counts())

660
FORWARD_ENTAILMENT     183
BACKWARD_ENTAILMENT    181
EQUIVALENCE            174
NEGATIVE_OTHER         122
Name: count, dtype: int64


In [12]:
# rows keep the same order as dev, so position i in pred_labels belongs to dev row i
preds_df = pd.DataFrame({"instance_id": dev["instance_id"], "label": pred_labels})
PRED_PATH = f"{RESULTS_DIR}/predictions_deberta_seed{SEED}.csv"
preds_df.to_csv(PRED_PATH, index=False)
print(PRED_PATH)
print(preds_df.head())

/content/SemEval-2027-Task-2-DiCo-NLI/results/predictions_deberta_seed13.csv
                             instance_id                label
0   dico_phrasis_0000002__en-en__flipped   FORWARD_ENTAILMENT
1  dico_phrasis_0000002__en-en__original  BACKWARD_ENTAILMENT
2   dico_phrasis_0000004__en-en__flipped   FORWARD_ENTAILMENT
3  dico_phrasis_0000004__en-en__original  BACKWARD_ENTAILMENT
4   dico_phrasis_0000008__en-en__flipped   FORWARD_ENTAILMENT


## Score with the official scorer

In [13]:
%cd /content/SemEval-2027-Task-2-DiCo-NLI
!python3 -m evaluation_functions \
  --gold final_data/dev/dico_nli_dev_track1_reference.csv \
  --predictions results/predictions_deberta_seed13.csv \
  --output-dir results/deberta_seed13

/content/SemEval-2027-Task-2-DiCo-NLI
{
  "classification": {
    "accuracy": 0.8196969696969697,
    "confusion_matrix": {
      "BACKWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 164,
        "EQUIVALENCE": 9,
        "FORWARD_ENTAILMENT": 2,
        "NEGATIVE_OTHER": 15
      },
      "EQUIVALENCE": {
        "BACKWARD_ENTAILMENT": 5,
        "EQUIVALENCE": 139,
        "FORWARD_ENTAILMENT": 6,
        "NEGATIVE_OTHER": 24
      },
      "FORWARD_ENTAILMENT": {
        "BACKWARD_ENTAILMENT": 2,
        "EQUIVALENCE": 11,
        "FORWARD_ENTAILMENT": 166,
        "NEGATIVE_OTHER": 11
      },
      "NEGATIVE_OTHER": {
        "BACKWARD_ENTAILMENT": 10,
        "EQUIVALENCE": 15,
        "FORWARD_ENTAILMENT": 9,
        "NEGATIVE_OTHER": 72
      }
    },
    "macro_f1": 0.8011517465189799,
    "per_label": {
      "BACKWARD_ENTAILMENT": {
        "f1": 0.8840970350404314,
        "false_negative": 26,
        "false_positive": 17,
        "label": "BACKWARD_ENTAILMENT",
       

In [14]:
!cat results/deberta_seed13/scores.txt

weighted_f1: 0.822789525121
soft_cons: 0.884476534296
hard_cons: 0.833935018051
